# Qualitative Analysis
This notebook was used to find transcription samples used in the qualitative analysis section of the thesis.

Acknowledgment: LLM assistance (Gemini) was used to refine the code structure and report formatting within this notebook.

### Quantitative Overview

In [1]:
import pandas as pd

# Load the evaluation results
csv_path = "asr_evaluation_big_model.csv"
df = pd.read_csv(csv_path)

In [2]:
total_samples = len(df)
improved = len(df[df['wer_improvement'] > 0])
worsened = len(df[df['wer_improvement'] < 0])
unchanged = len(df[df['wer_improvement'] == 0])

summary_data = {
    "Impact Category": ["Improved by KenLM", "Worsened by KenLM", "No Change / Identical"],
    "Sample Count": [improved, worsened, unchanged],
    "Percentage (%)": [(improved/total_samples)*100, (worsened/total_samples)*100, (unchanged/total_samples)*100]
}

summary_df = pd.DataFrame(summary_data)
print("### QUANTITATIVE OVERVIEW TABLE ###")
display(summary_df.round(2))

# Average error profile comparison
print("\n### AVERAGE WORD-LEVEL ALIGNMENT PROFILE ###")
alignment_comparison = df[['greedy_substitutions', 'kenlm_substitutions', 
                           'greedy_insertions', 'kenlm_insertions', 
                           'greedy_deletions', 'kenlm_deletions']].mean().to_frame(name='Dataset Mean')
display(alignment_comparison.round(3))

### QUANTITATIVE OVERVIEW TABLE ###


,Impact Category,Sample Count,Percentage (%)
0,Improved by KenLM,302,16.04
1,Worsened by KenLM,123,6.53
2,No Change / Identical,1458,77.43



### AVERAGE WORD-LEVEL ALIGNMENT PROFILE ###


,Dataset Mean
greedy_substitutions,0.853
kenlm_substitutions,0.695
greedy_insertions,0.101
kenlm_insertions,0.122
greedy_deletions,0.148
kenlm_deletions,0.131


### Qualitative Report Generator

In [3]:
def run_qualitative_analysis(df, output_report_path="qualitative_report.md"):
    """
    Slices the inference dataframe by domains, tracks 100% correct (Zero-WER) metrics,
    and extracts robust behavioral profiles for dissertation write-ups.
    """
    print("\n--- Running Comprehensive Qualitative Analysis ---")
    
    # Pre-calculate composite metrics
    df['kenlm_avg_error'] = (df['kenlm_wer'] + df['kenlm_cer']) / 2.0
    
    domains = df['source_domain'].unique()
    markdown_report = "# 📊 Qualitative Error Analysis & Diagnostic Report\n\n"
    
    for domain in domains:
        df_dom = df[df['source_domain'] == domain].copy()
        total_samples = len(df_dom)
        mean_domain_wer = df_dom['kenlm_wer'].mean()
        
        # --- 100% Correct (Zero-WER) Calculations ---
        greedy_perfect = (df_dom['greedy_wer'] == 0).sum()
        kenlm_perfect = (df_dom['kenlm_wer'] == 0).sum()
        
        greedy_perfect_pct = (greedy_perfect / total_samples) * 100
        kenlm_perfect_pct = (kenlm_perfect / total_samples) * 100
        
        # Specific decoder interactions
        rescued_by_kenlm = df_dom[(df_dom['greedy_wer'] > 0) & (df_dom['kenlm_wer'] == 0)]
        damaged_by_kenlm = df_dom[(df_dom['greedy_wer'] == 0) & (df_dom['kenlm_wer'] > 0)]
        
        # --- Build Domain Header & Quantitative Summary ---
        markdown_report += f"## 🌍 Test Set / Domain: {domain}\n"
        markdown_report += f"*Total samples: {total_samples} | Domain Mean KenLM WER: {mean_domain_wer:.4f}*\n\n"
        
        markdown_report += "### 🎯 Perfect Transcription (Zero-WER) Summary\n"
        markdown_report += "| Decoder | Perfect Sentences (Count) | Perfect Sentences (%) |\n"
        markdown_report += "| :--- | :---: | :---: |\n"
        markdown_report += f"| **Greedy Baseline** | {greedy_perfect} / {total_samples} | {greedy_perfect_pct:.2f}% |\n"
        markdown_report += f"| **With KenLM** | {kenlm_perfect} / {total_samples} | {kenlm_perfect_pct:.2f}% |\n\n"
        markdown_report += f"* KenLM successfully **rescued** `{len(rescued_by_kenlm)}` error-prone greedy sentences to 100% accuracy.\n"
        markdown_report += f"* KenLM accidentally **damaged** `{len(damaged_by_kenlm)}` previously perfect sentences.\n\n"
        
        # --- Structured Profiles for Write-up Selection ---
        profiles = {
            "🏆 1. Best Performance Pool (Top candidates for 'Perfect Baseline Case')": 
                df_dom.nsmallest(10, 'kenlm_avg_error'),
                
            "💥 2. Worst Performance Pool (Top candidates for 'Catastrophe Case')": 
                df_dom.nlargest(3, 'kenlm_avg_error'),
                
            "📉 3. Mean Statistical Samples (Closest to Domain Mean WER)": 
                df_dom.iloc[(df_dom['kenlm_wer'] - mean_domain_wer).abs().argsort()[:3]],
                
            "✨ 4. KenLM Rescues (Acoustic typos successfully corrected by language model)": 
                rescued_by_kenlm.head(3),
                
            "💔 5. KenLM Regressions (Linguistic over-corrections that ruined perfect text)": 
                damaged_by_kenlm.sort_values('wer_improvement', ascending=True).head(10),
                
            "🔍 6. High CER but Low WER (Phonetic shifts / Orthographic variations)": 
                df_dom[(df_dom['kenlm_cer'] > 0.25) & (df_dom['kenlm_wer'] < 0.15)].head(3),
                
            "🧠 7. Acoustic Triumph, Decoder Failure (Overly punitive word matching - Low CER / High WER)": 
                df_dom[(df_dom['kenlm_cer'] < 0.08) & (df_dom['kenlm_wer'] > 0.35)].head(3)
        }
        
        # Format profiles into Markdown
        for title, sampled_df in profiles.items():
            markdown_report += f"### {title}\n"
            if sampled_df.empty:
                markdown_report += "_No samples found matching these specific mathematical bounds in this domain._\n\n"
                continue
                
            for idx, (_, row) in enumerate(sampled_df.iterrows(), 1):
                markdown_report += f"**Sample #{idx} | File:** `{row['filename']}`\n"
                markdown_report += f"* **Ground Truth:** \"{row['ground_truth']}\"\n"
                markdown_report += f"* **Greedy Pred:** \"{row['greedy_pred']}\" (WER: {row['greedy_wer']:.2f}, CER: {row['greedy_cer']:.2f})\n"
                markdown_report += f"* **KenLM Pred:** \"{row['kenlm_pred']}\" (WER: {row['kenlm_wer']:.2f}, CER: {row['kenlm_cer']:.2f})\n"
                markdown_report += f"* **Metrics:** Δ WER Improvement: {row['wer_improvement']:.2f} | Error Composite: {row['kenlm_avg_error']:.2f}\n"
                markdown_report += f"  * Total Edits -> Subs: {row['kenlm_substitutions']}, Ins: {row['kenlm_insertions']}, Del: {row['kenlm_deletions']}\n\n"
        
        markdown_report += "---\n\n"
        
    # Write report to disk
    with open(output_report_path, "w", encoding="utf-8") as f:
        f.write(markdown_report)
        
    print(f"🔬 Comprehensive diagnostic report generated successfully at: {output_report_path}")

In [4]:
import pandas as pd

if __name__ == "__main__":
    # Load the saved CSV file back into a DataFrame object
    df_results = pd.read_csv("asr_evaluation_big_model.csv")
    
    # Run the qualitative analysis on it
    run_qualitative_analysis(df_results, output_report_path="qualitative_report.md")


--- Running Comprehensive Qualitative Analysis ---
🔬 Comprehensive diagnostic report generated successfully at: qualitative_report.md
